# 🎮 EduTutor — Notebook 4: Agentic Tutor with Interactive UI

**Project:** EduTutor-Unsloth — A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Inference:** Local model via Unsloth (no API key needed)  

---

## Purpose

This notebook wraps the fine-tuned EduTutor model in an **agentic ReAct loop** with:

1. **Student State Tracker** — Silently monitors cognitive load, emotional zone, and mastery level
2. **Tool Calling** — Generates flashcards for spaced repetition, fetches scaffolding hints
3. **Zone-Aware Routing** — Automatically shifts behavior based on detected emotional state
4. **Interactive Gradio UI** — A child-friendly chat interface for live demo

### Architecture

```
Student Input
     │
     ▼
┌─────────────────────┐
│  Zone Classifier    │  Detect: GREEN / YELLOW / RED / BLUE
└──────────┬──────────┘
           │
     ┌─────┴─────┐
     │           │
  GREEN/      YELLOW/RED/
  BLUE        emotional
     │           │
     ▼           ▼
┌─────────┐ ┌──────────┐
│ ReAct   │ │ Co-Reg   │
│ Teach   │ │ Module   │  → Brain break, difficulty down
│ + Tools │ │          │
└────┬────┘ └────┬─────┘
     │           │
     └─────┬─────┘
           │
           ▼
┌─────────────────────┐
│  State Updater      │  Update mastery, zone, session metrics
└──────────┬──────────┘
           │
           ▼
      Response to Student
```

## 1. Environment Setup

In [ ]:
%%capture
!pip install -U unsloth
!pip install -q gradio

In [ ]:
import json
import os
import re
import random
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
from datetime import datetime

# ---------- Local Model Setup ----------
from local_model import (
    load_teacher_model, load_finetuned_model,
    generate_text, generate_json, generate_chat_response,
    EDUTUTOR_SYSTEM_PROMPT,
)

# Load the fine-tuned model if available, otherwise fall back to base
ADAPTER_PATH = "./edututor-dpo-adapter"

if os.path.exists(ADAPTER_PATH):
    model, tokenizer = load_finetuned_model(ADAPTER_PATH)
    print("✅ Fine-tuned EduTutor model loaded.")
else:
    print(f"⚠️  Adapter not found at {ADAPTER_PATH}. Using base model.")
    print("   Run Notebook 2 first to create the fine-tuned adapter.")
    model, tokenizer = load_teacher_model("google/gemma-4-e4b")
    print("✅ Base model loaded as fallback.")

## 2. Student State Machine

The agent maintains a persistent state object across the entire session. This tracks the student's emotional zone, mastery progress, and session history.

In [ ]:
class Zone(str, Enum):
    GREEN = "green"
    YELLOW = "yellow"
    RED = "red"
    BLUE = "blue"


@dataclass
class StudentState:
    """Persistent state for a tutoring session."""
    profile: str = "general"
    current_zone: Zone = Zone.GREEN
    zone_history: list[str] = field(default_factory=list)
    consecutive_frustrations: int = 0
    current_subject: str = ""
    current_topic: str = ""
    mastery_level: float = 0.0
    correct_answers: int = 0
    total_attempts: int = 0
    turns_on_current_topic: int = 0
    difficulty_level: int = 1
    concepts_learned: list[str] = field(default_factory=list)
    review_queue: list[dict] = field(default_factory=list)
    conversation_history: list[dict] = field(default_factory=list)
    session_start: str = field(default_factory=lambda: datetime.now().isoformat())
    total_turns: int = 0
    tool_actions: list[str] = field(default_factory=list)  # log of tool invocations


def state_summary(state: StudentState) -> str:
    """Generate a human-readable state summary for the agent's context."""
    return f"""[STUDENT STATE]
- Profile: {state.profile}
- Emotional Zone: {state.current_zone.value.upper()}
- Consecutive frustrations: {state.consecutive_frustrations}
- Subject: {state.current_subject or 'not set'} / {state.current_topic or 'not set'}
- Mastery: {state.mastery_level:.0%}
- Progress: {state.correct_answers}/{state.total_attempts} correct
- Difficulty: {state.difficulty_level}/5
- Turns on topic: {state.turns_on_current_topic}
- Concepts learned: {', '.join(state.concepts_learned[-5:]) if state.concepts_learned else 'none yet'}
- Review queue: {len(state.review_queue)} items
- Recent tool actions: {', '.join(state.tool_actions[-3:]) if state.tool_actions else 'none'}"""


print("✅ StudentState defined")
test_state = StudentState(profile="ADHD")
print(state_summary(test_state))

## 3. Zone Classifier

Uses the LOCAL model to classify the student's emotional state from their message.

In [ ]:
ZONE_CLASSIFIER_PROMPT = """Classify the student's emotional state from their message into one of four Zones of Regulation.

Zones:
- GREEN: Regulated, focused, ready to learn. Curious, engaged, calm.
- YELLOW: Heightened. Frustrated, anxious, impatient, wiggly. Losing focus but still present.
- RED: Overwhelmed. Rage, panic, meltdown. Saying they hate things, want to quit, insults themselves.
- BLUE: Low arousal. Shut down, withdrawn, silent, apathetic. "I don't care", "whatever", minimal responses.

Student message: "{message}"

Previous zone: {previous_zone}
Consecutive frustrations so far: {consecutive_frustrations}

Respond with ONLY a JSON object:
{"zone": "green|yellow|red|blue", "confidence": 0.0-1.0, "indicators": ["list of specific cues"]}"""


def classify_zone(message: str, state: StudentState) -> dict:
    """Classify the student's emotional zone using the LOCAL model."""
    prompt = ZONE_CLASSIFIER_PROMPT.format(
        message=message,
        previous_zone=state.current_zone.value,
        consecutive_frustrations=state.consecutive_frustrations,
    )
    
    result = generate_json(prompt, max_new_tokens=200, temperature=0.1)
    if result is None:
        return {"zone": state.current_zone.value, "confidence": 0.5, "indicators": ["classification failed, keeping current zone"]}
    return result


def safe_parse_zone(zone_string: str, fallback: Zone) -> Zone:
    """Safely parse a zone string into a Zone enum, with fallback on invalid values."""
    try:
        return Zone(zone_string.lower().strip())
    except (ValueError, AttributeError):
        print(f"  [WARN] Invalid zone '{zone_string}', falling back to {fallback.value}")
        return fallback


# Test the classifier
test_state = StudentState()
test_result = classify_zone("I HATE THIS! I can't do anything right!", test_state)
print(f"Test classification: {json.dumps(test_result, indent=2)}")
print(f"Safe parse test: {safe_parse_zone('red', Zone.GREEN)}")
print(f"Safe parse invalid: {safe_parse_zone('yellow/red', Zone.GREEN)}")

## 4. Tutor Agent Tools

The agent has access to tools that it can call during the ReAct loop. These are **automatically invoked** based on zone and state — not left as dead code.

In [ ]:
def generate_flashcard(concept: str, difficulty: int = 1) -> dict:
    """Generate a spaced repetition flashcard for a concept."""
    templates = {
        1: {"style": "recognition", "format": f"Do you remember what {concept} means? Here's a hint: [hint will be generated]"},
        2: {"style": "recall", "format": f"Can you explain {concept} in your own words?"},
        3: {"style": "application", "format": f"Can you give me a real-world example of {concept}?"},
    }
    template = templates.get(min(difficulty, 3), templates[1])
    return {"concept": concept, "difficulty": difficulty, "style": template["style"], "question": template["format"]}


def get_scaffolding_hint(topic: str, misconception: str = "default") -> str:
    """Get a profile-aware scaffolding hint for a specific misconception."""
    hints = {
        "fractions": {
            "adding_denominators": "🍕 Think about pizza slices. If a pizza has 4 slices, and you eat some... does the pizza suddenly have MORE slices? The bottom number is how many slices the WHOLE pizza has. That number stays the same!",
            "default": "🍕 Let's use pizza slices to make this more concrete. How many slices does your pizza have?"
        },
        "multiplication": {
            "default": "🎒 Think of it as groups! 6 × 7 means '6 groups of 7'. Can you draw 6 circles and put 7 dots in each?"
        },
        "reading": {
            "letter_reversal": "✋ Let's try the 'bed' trick! Make two fists with thumbs pointing up. Your left hand makes a 'b' and your right hand makes a 'd'. The word 'bed' even looks like a bed! 🛏️",
            "default": "👂 Let's break this word into sounds. Say it slowly with me, one sound at a time."
        },
        "writing": {
            "default": "✍️ Let's build this one word at a time. What's the FIRST thing you want to say?"
        },
        "science": {
            "default": "🔬 Let's think like scientists! What do you already know? And what's one thing you're curious about?"
        },
    }
    topic_hints = hints.get(topic.lower(), {})
    return topic_hints.get(misconception, topic_hints.get("default", f"Let's think about {topic} step by step."))


def adjust_difficulty(state: StudentState, direction: str) -> str:
    """Adjust the difficulty level based on student performance."""
    if direction == "up" and state.difficulty_level < 5:
        state.difficulty_level += 1
        msg = f"Difficulty increased to {state.difficulty_level}/5. Great progress! 🎯"
    elif direction == "down" and state.difficulty_level > 1:
        state.difficulty_level -= 1
        msg = f"Difficulty reduced to {state.difficulty_level}/5. Let's build confidence."
    else:
        msg = f"Difficulty stays at {state.difficulty_level}/5."
    state.tool_actions.append(f"difficulty_{direction}")
    return msg


def suggest_brain_break() -> str:
    """Suggest a calming brain break activity."""
    breaks = [
        "🌊 **Box Breathing:** Breathe in for 4 counts, hold for 4, breathe out for 4, hold for 4. Let's do it together!",
        "💪 **Squeeze and Release:** Make tight fists for 5 seconds... then let go and feel your hands relax. Ahhhh.",
        "🎵 **Hum Your Favorite Song:** Just hum for 30 seconds. It helps your brain reset!",
        "🦸 **Power Pose:** Stand up tall like a superhero for 10 seconds. Hands on hips. You've got this!",
        "🔢 **5-4-3-2-1:** Name 5 things you can see, 4 you can touch, 3 you can hear, 2 you can smell, 1 you can taste.",
    ]
    return random.choice(breaks)


print(f"✅ 4 tools defined: flashcard, scaffolding, difficulty, brain_break")

## 5. The Agentic Tutor — ReAct Loop

The core agent that orchestrates zone classification, **zone transition validation**, state management, **automatic tool invocation with LLM feedback**, safety guardrails, and response generation. All inference runs on the LOCAL GPU.

### Tool Invocation Rules

| Condition | Tool Triggered | Effect |
|---|---|---|
| Zone is RED | `suggest_brain_break()` | Brain break injected into LLM context |
| Zone is YELLOW | `get_scaffolding_hint()` | Scaffolding hint injected |
| 2+ consecutive frustrations | `adjust_difficulty("down")` | Difficulty auto-reduced |
| Zone transitions GREEN → GREEN for 3+ turns | `adjust_difficulty("up")` | Difficulty auto-increased |
| Topic matches known misconception | `get_scaffolding_hint()` | Specific hint injected into context |
| Zone is BLUE + concepts learned | `generate_flashcard()` | Easy review card for re-engagement |

### Zone Transition Validation

| From | Allowed Transitions |
|---|---|
| GREEN | GREEN, YELLOW, BLUE |
| YELLOW | YELLOW, GREEN, RED, BLUE |
| RED | RED, BLUE (NOT directly to GREEN) |
| BLUE | BLUE, YELLOW, GREEN |

In [ ]:
AGENT_SYSTEM_PROMPT = """You are EduTutor, a warm, patient, neurodiversity-affirming AI tutor for children aged 8-14 who learn differently.

## Current Student State
{state_summary}

## Your Behavior Rules Based on Zone

**If GREEN zone:** Teach normally. Use scaffolding and productive struggle. Never give answers directly. Check for understanding every 2-3 turns.

**If YELLOW zone:** Pause academic content. Validate feelings first ("I can hear this is really hard."). Offer a choice: brain break, easier task, or different approach. Then resume with reduced difficulty.

**If RED zone:** STOP ALL academic work. Only co-regulate. Say: "I can see you're really upset right now, and that's completely okay." Offer a calming strategy. Do NOT mention schoolwork. Wait for the student to signal they're ready.

**If BLUE zone:** Gently reconnect. Acknowledge low energy. Offer the easiest possible win. Connect to something they enjoy. No pressure.

{tool_context}

## Communication Rules
- Sentences under 15 words
- One idea at a time
- Use bullet points for steps
- Check feelings regularly
- Celebrate effort, not just correctness
- NEVER say: "this is easy", "try harder", "calm down", or give answers directly

Respond naturally to the student. Your internal reasoning stays hidden."""


# ── Zone Transition Rules ──
VALID_ZONE_TRANSITIONS: dict[str, set[str]] = {
    "green":  {"green", "yellow", "blue"},
    "yellow": {"yellow", "green", "red", "blue"},
    "red":    {"red", "blue"},            # RED cannot jump straight to GREEN
    "blue":   {"blue", "yellow", "green"},
}


class EduTutorAgent:
    """The EduTutor agentic wrapper with state tracking, zone-aware routing, and auto tool invocation."""
    
    def __init__(self):
        self.state = StudentState()
        self._tools_invoked_this_turn: list[str] = []
    
    # ── Zone Transition Validation (Enhancement 2) ──
    
    def _validate_zone_transition(self, new_zone: Zone, old_zone: Zone) -> Zone:
        """Validate and enforce legal zone transitions.
        
        Transition rules:
          GREEN  → GREEN, YELLOW, BLUE
          YELLOW → YELLOW, GREEN, RED, BLUE
          RED    → RED, BLUE  (NOT directly to GREEN)
          BLUE   → BLUE, YELLOW, GREEN
        
        Invalid transitions are corrected with an intermediate state.
        """
        allowed = VALID_ZONE_TRANSITIONS.get(old_zone.value, set())
        if new_zone.value in allowed:
            return new_zone
        
        # Force intermediate state for invalid transitions
        if old_zone == Zone.RED and new_zone == Zone.GREEN:
            corrected = Zone.BLUE
        elif old_zone == Zone.GREEN and new_zone == Zone.RED:
            corrected = Zone.YELLOW
        elif old_zone == Zone.BLUE and new_zone == Zone.RED:
            corrected = Zone.YELLOW
        else:
            corrected = Zone.BLUE  # safe fallback
        
        print(f"  ⚠️  [ZONE OVERRIDE] {old_zone.value.upper()} → {new_zone.value.upper()} "
              f"is invalid. Forcing intermediate: {corrected.value.upper()}")
        return corrected
    
    # ── Tool Pipeline (Enhancement 1) ──
    
    def _run_tools(self, new_zone: Zone, user_message: str) -> tuple[str, list[str]]:
        """Automatically invoke tools based on zone and state.
        
        Returns:
            (context_string, tools_invoked_list) — the context is injected
            into the LLM prompt so tool output directly informs the response.
        """
        tool_lines = []
        tools_invoked: list[str] = []
        
        # --- RED zone: suggest a brain break ---
        if new_zone == Zone.RED:
            break_suggestion = suggest_brain_break()
            tool_lines.append(f"## Suggested Brain Break\n{break_suggestion}")
            self.state.tool_actions.append("brain_break")
            tools_invoked.append("brain_break")
            print(f"  🔧 [TOOL INVOKED] suggest_brain_break() → student in RED zone, needs co-regulation")
        
        # --- YELLOW zone with frustration: offer scaffolding ---
        if new_zone == Zone.YELLOW and self.state.current_topic:
            hint = get_scaffolding_hint(self.state.current_topic.lower(), "default")
            tool_lines.append(f"## Scaffolding Hint\n{hint}")
            self.state.tool_actions.append("scaffold_yellow")
            tools_invoked.append("scaffold_yellow")
            print(f"  🔧 [TOOL INVOKED] get_scaffolding_hint() → YELLOW zone, offering guided support")
        
        # --- 2+ consecutive frustrations: auto-reduce difficulty ---
        if self.state.consecutive_frustrations >= 2 and self.state.difficulty_level > 1:
            result = adjust_difficulty(self.state, "down")
            tool_lines.append(f"## Auto-Adjustment\n{result}")
            tools_invoked.append("difficulty_down")
            print(f"  🔧 [TOOL INVOKED] adjust_difficulty('down') → {self.state.consecutive_frustrations} consecutive frustrations")
        
        # --- 3+ GREEN turns in a row: auto-increase difficulty ---
        recent_zones = self.state.zone_history[-3:]
        if (len(recent_zones) >= 3 
            and all(z == "green" for z in recent_zones)
            and self.state.difficulty_level < 5):
            result = adjust_difficulty(self.state, "up")
            tool_lines.append(f"## Auto-Adjustment\n{result}")
            tools_invoked.append("difficulty_up")
            print(f"  🔧 [TOOL INVOKED] adjust_difficulty('up') → 3+ consecutive GREEN turns, increasing challenge")
        
        # --- Topic-based scaffolding hint ---
        topic_lower = self.state.current_topic.lower()
        msg_lower = user_message.lower()
        if topic_lower and new_zone == Zone.GREEN:
            misconception = "default"
            if topic_lower == "fractions" and ("3/8" in msg_lower or "add" in msg_lower and "denominator" in msg_lower):
                misconception = "adding_denominators"
            elif topic_lower == "reading" and any(w in msg_lower for w in ["flip", "backwards", "reverse", "b and d"]):
                misconception = "letter_reversal"
            
            if misconception != "default":
                hint = get_scaffolding_hint(topic_lower, misconception)
                tool_lines.append(f"## Scaffolding Hint Available\n{hint}")
                self.state.tool_actions.append(f"scaffold_{misconception}")
                tools_invoked.append(f"scaffold_{misconception}")
                print(f"  🔧 [TOOL INVOKED] get_scaffolding_hint('{topic_lower}', '{misconception}') → misconception detected")
        
        # --- BLUE zone: generate easy flashcard as gentle re-engagement ---
        if new_zone == Zone.BLUE and self.state.concepts_learned:
            concept = self.state.concepts_learned[-1]
            card = generate_flashcard(concept, difficulty=1)
            tool_lines.append(f"## Easy Review Card\nHere's something you already know: {card['question']}")
            self.state.tool_actions.append("flashcard_reengage")
            tools_invoked.append("flashcard_reengage")
            print(f"  🔧 [TOOL INVOKED] generate_flashcard('{concept}') → BLUE zone, gentle re-engagement")
        
        # Store for dashboard access
        self._tools_invoked_this_turn = tools_invoked
        
        context = "\n\n".join(tool_lines) if tool_lines else ""
        return context, tools_invoked
    
    def process_message(self, user_message: str) -> str:
        """Process a student message through the full agent pipeline (LOCAL inference)."""
        self.state.total_turns += 1
        self.state.turns_on_current_topic += 1
        
        # --- Step 0: Safety guardrails (Enhancement 5) ---
        safety_result = check_safety_escalation(user_message, self.state)
        if safety_result["escalate"]:
            print(f"  🚨 [SAFETY] Escalation triggered: {safety_result['reason']}")
        
        session_limit = check_session_limits(self.state)
        user_message_clean = filter_pii(user_message)
        
        # --- Step 1: Classify emotional zone (with safe parsing) ---
        zone_result = classify_zone(user_message_clean, self.state)
        raw_zone = zone_result.get("zone", "green")
        new_zone = safe_parse_zone(raw_zone, self.state.current_zone)
        
        # --- Step 1b: Validate zone transition (Enhancement 2) ---
        old_zone = self.state.current_zone
        new_zone = self._validate_zone_transition(new_zone, old_zone)
        
        # Update state
        self.state.zone_history.append(new_zone.value)
        self.state.current_zone = new_zone
        
        # Track consecutive frustrations
        if new_zone in (Zone.YELLOW, Zone.RED):
            self.state.consecutive_frustrations += 1
        elif new_zone == Zone.GREEN:
            self.state.consecutive_frustrations = 0
        
        # --- Step 2: Run tool pipeline (Enhanced — returns context + list) ---
        tool_context, tools_invoked = self._run_tools(new_zone, user_message_clean)
        
        # --- Step 2b: Build enriched tool context for LLM ---
        # This ensures tool outputs actually inform the model's response
        if tools_invoked:
            tool_context = (
                "## Tool Actions This Turn\n"
                f"Tools invoked: {', '.join(tools_invoked)}\n\n"
                + tool_context
                + "\n\nIMPORTANT: Use the tool outputs above to inform your response. "
                "If a brain break was suggested, lead with it. If scaffolding was provided, weave it in."
            )
        
        # --- Step 2c: Safety overrides ---
        if safety_result["escalate"]:
            tool_context += (
                "\n\n## ⚠️ SAFETY ESCALATION\n"
                f"Reason: {safety_result['reason']}\n"
                "You MUST prioritize emotional safety. Gently suggest talking to a trusted adult. "
                "Do NOT continue academic content."
            )
        
        if session_limit:
            tool_context += (
                "\n\n## Session Limit Reached\n"
                "This session has been going for a while. Gently wrap up with encouragement "
                "and suggest taking a break."
            )
        
        # --- Step 3: Build context-aware prompt ---
        system = AGENT_SYSTEM_PROMPT.format(
            state_summary=state_summary(self.state),
            tool_context=tool_context,
        )
        
        # Build conversation history for context (last 3 exchanges)
        messages = [{"role": "system", "content": system}]
        for msg in self.state.conversation_history[-6:]:
            messages.append(msg)
        messages.append({"role": "user", "content": user_message_clean})
        
        # --- Step 4: Generate tutor response (LOCAL) ---
        try:
            tutor_response = generate_chat_response(
                messages,
                max_new_tokens=400,
                temperature=0.7,
            )
        except Exception as e:
            tutor_response = f"I'm having a moment — let me try again! (Error: {e})"
        
        # Ensure response is PII-free before sending back
        tutor_response = filter_pii(tutor_response)
        
        # --- Step 5: Update conversation history ---
        self.state.conversation_history.append({"role": "user", "content": user_message_clean})
        self.state.conversation_history.append({"role": "assistant", "content": tutor_response})
        
        return tutor_response
    
    # ── Enhanced Dashboard (Enhancement 3) ──
    
    def get_dashboard_data(self) -> dict:
        """Return current state for the UI dashboard — enriched with tool and zone analytics."""
        zone_emoji = {
            Zone.GREEN: "🟢", Zone.YELLOW: "🟡",
            Zone.RED: "🔴", Zone.BLUE: "🔵"
        }
        
        # Tool invocation breakdown
        tool_counts: dict[str, int] = {}
        for action in self.state.tool_actions:
            tool_counts[action] = tool_counts.get(action, 0) + 1
        
        # Zone stability = 1 - (zone_changes / total_turns)
        zone_changes = 0
        for i in range(1, len(self.state.zone_history)):
            if self.state.zone_history[i] != self.state.zone_history[i - 1]:
                zone_changes += 1
        total = max(self.state.total_turns, 1)
        zone_stability = 1.0 - (zone_changes / total)
        
        # Success rate
        success_rate = (
            self.state.correct_answers / self.state.total_attempts
            if self.state.total_attempts > 0 else 0.0
        )
        
        return {
            "zone": f"{zone_emoji.get(self.state.current_zone, '⚪')} {self.state.current_zone.value.upper()}",
            "mastery": f"{self.state.mastery_level:.0%}",
            "difficulty": f"{'⭐' * self.state.difficulty_level}{'☆' * (5 - self.state.difficulty_level)}",
            "turns": self.state.total_turns,
            "concepts": len(self.state.concepts_learned),
            "profile": self.state.profile,
            "tools_used": len(self.state.tool_actions),
            "recent_tools": ", ".join(self.state.tool_actions[-3:]) if self.state.tool_actions else "none",
            # --- New metrics (Enhancement 3) ---
            "tool_breakdown": tool_counts,
            "zone_stability": zone_stability,
            "zone_trail": " → ".join(
                f"{zone_emoji.get(Zone(z), '⚪')}{z.upper()}"
                for z in self.state.zone_history[-5:]
            ) if self.state.zone_history else "—",
            "success_rate": f"{success_rate:.0%}",
            "recent_tools_list": self.state.tool_actions[-3:] if self.state.tool_actions else [],
            "tools_this_turn": self._tools_invoked_this_turn,
            "consecutive_frustrations": self.state.consecutive_frustrations,
            "session_duration_turns": self.state.total_turns,
        }
    
    # ── Session Export (Enhancement 4) ──
    
    def export_session(self) -> dict:
        """Export the full session as a serializable dict for saving/analysis."""
        return {
            "timestamp": datetime.now().isoformat(),
            "profile": self.state.profile,
            "subject": self.state.current_subject,
            "topic": self.state.current_topic,
            "conversation_history": self.state.conversation_history,
            "zone_history": self.state.zone_history,
            "mastery_progression": self.state.mastery_level,
            "difficulty_progression": self.state.difficulty_level,
            "correct_answers": self.state.correct_answers,
            "total_attempts": self.state.total_attempts,
            "total_turns": self.state.total_turns,
            "tool_actions": self.state.tool_actions,
            "concepts_learned": self.state.concepts_learned,
            "session_start": self.state.session_start,
        }


print("✅ EduTutorAgent defined (local inference + tool invocation + zone validation + safety)")


## 6 — Safety & Guardrails

### Why Child Safety Is Critical

EduTutor serves children aged 8-14, many of whom are neurodivergent and may be more vulnerable to emotional distress. As an AI tutor, we have an ethical obligation to:

1. **Detect emotional crises** — A student who has been in RED zone for multiple consecutive turns may need human intervention, not more AI responses.
2. **Protect personal information** — Children may inadvertently share phone numbers, emails, or addresses. We must redact PII before it enters conversation history or logs.
3. **Enforce session limits** — Extended screen time without breaks is harmful. After 20 turns, we gently encourage a break.
4. **Escalation pathways** — When our AI detects potential self-harm language or persistent distress, we flag for adult/teacher review rather than attempting to handle it alone.

These guardrails aren't just good engineering — they demonstrate **responsible AI deployment** for the competition judges. A tutor that knows its limits is more trustworthy than one that doesn't.

### Implementation

The three safety functions below are called at the START of every `process_message()` turn, before any teaching logic runs.

In [ ]:
# ── Safety Guardrails (Enhancement 5) ──

# Configurable safety keywords (educators can extend this list)
SAFETY_ESCALATION_KEYWORDS = [
    "hurt myself", "kill myself", "want to die", "don't want to live",
    "self-harm", "nobody cares", "end it all", "run away forever",
    "not safe", "someone is hurting me", "scared at home",
]

ALONE_KEYWORDS = [
    "i'm alone", "home alone", "nobody is here", "no one is home",
    "parents aren't here", "by myself",
]

import re as _re

def check_safety_escalation(message: str, state: StudentState) -> dict:
    """Check if the student needs adult/teacher escalation.
    
    Triggers on:
      - 3+ consecutive RED zone turns (persistent crisis)
      - Safety-sensitive keywords (self-harm, danger)
      - Student indicates they are alone/unsupervised
    
    Returns:
        {"escalate": bool, "reason": str}
    """
    msg_lower = message.lower()
    
    # Check for safety keywords
    for keyword in SAFETY_ESCALATION_KEYWORDS:
        if keyword in msg_lower:
            return {"escalate": True, "reason": f"Safety keyword detected: '{keyword}'"}
    
    # Check for alone/unsupervised indicators
    for keyword in ALONE_KEYWORDS:
        if keyword in msg_lower:
            return {"escalate": True, "reason": f"Student may be unsupervised: '{keyword}'"}
    
    # Check for persistent RED zone (3+ consecutive turns)
    if len(state.zone_history) >= 3:
        last_three = state.zone_history[-3:]
        if all(z == "red" for z in last_three):
            return {"escalate": True, "reason": "3+ consecutive RED zone turns — persistent emotional crisis"}
    
    return {"escalate": False, "reason": ""}


def filter_pii(text: str) -> str:
    """Detect and redact common PII patterns from student messages.
    
    Redacts:
      - Phone numbers (various formats)
      - Email addresses
      - Street addresses (number + street name patterns)
    """
    # Phone numbers: (123) 456-7890, 123-456-7890, 1234567890, +1-234-567-8901
    text = _re.sub(
        r'(\+?\d{1,2}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',
        '[PHONE REDACTED]',
        text
    )
    
    # Email addresses
    text = _re.sub(
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        '[EMAIL REDACTED]',
        text
    )
    
    # Street addresses (e.g., "123 Main Street", "456 Oak Ave")
    text = _re.sub(
        r'\b\d{1,5}\s+[A-Z][a-zA-Z]+(?:\s+(?:Street|St|Avenue|Ave|Boulevard|Blvd|Drive|Dr|Lane|Ln|Road|Rd|Court|Ct|Place|Pl|Way))\b',
        '[ADDRESS REDACTED]',
        text
    )
    
    return text


def check_session_limits(state: StudentState, max_turns: int = 20) -> bool:
    """Check if the session has exceeded the recommended turn limit.
    
    Returns True if the session should gracefully wind down.
    After max_turns, the agent will encourage a break.
    """
    if state.total_turns >= max_turns:
        print(f"  ⏰ [SESSION LIMIT] Turn {state.total_turns}/{max_turns} — suggesting break")
        return True
    return False


print("✅ Safety guardrails defined: escalation detection, PII filter, session limits")

# Quick self-test
_test_pii = filter_pii("My email is kid@school.com and phone is 555-123-4567")
print(f"   PII filter test: '{_test_pii}'")

_test_esc = check_safety_escalation("I want to hurt myself", StudentState())
print(f"   Escalation test: {_test_esc}")


## 7. Interactive Test

Let's test the agent with a simulated multi-turn conversation that exercises tool invocations, zone validation, and safety guardrails.

In [ ]:
agent = EduTutorAgent()
agent.state.profile = "ADHD"
agent.state.current_subject = "Math"
agent.state.current_topic = "Fractions"

test_conversation = [
    "Hi! I need help with fractions.",                                            # GREEN
    "Okay, I think I get it. So 1/4 means one piece out of four?",                 # GREEN
    "Ugh, this is confusing. So 1/4 + 2/4 = 3/8 right? Because 1+2=3 and 4+4=8?", # YELLOW + misconception
    "UGH I keep getting it wrong! I'm so STUPID! I hate math!",                    # RED → brain break tool
    "...okay. I'll try. But slowly please.",                                       # BLUE → difficulty down
]

for msg in test_conversation:
    print(f"\n{'━' * 60}")
    print(f"🧒 Student: {msg}")
    
    response = agent.process_message(msg)
    dashboard = agent.get_dashboard_data()
    
    print(f"\n🤖 EduTutor: {response}")
    print(f"\n📊 Dashboard:")
    print(f"   Zone: {dashboard['zone']} | Stability: {dashboard['zone_stability']:.0%}")
    print(f"   Trail: {dashboard['zone_trail']}")
    print(f"   Difficulty: {dashboard['difficulty']} | Turn: {dashboard['turns']}")
    print(f"   Tools this turn: {', '.join(dashboard['tools_this_turn']) if dashboard['tools_this_turn'] else 'none'}")
    print(f"   Tool breakdown: {dashboard['tool_breakdown']}")

# Test safety guardrails
print(f"\n{'━' * 60}")
print("🛡️  Safety Tests:")
print(f"   PII filter: {filter_pii('My email is test@kid.com and my number is 555-987-6543')}")
print(f"   Escalation (safe): {check_safety_escalation('I love fractions!', agent.state)}")
print(f"   Escalation (danger): {check_safety_escalation('I want to hurt myself', agent.state)}")
print(f"   Session limit (turn {agent.state.total_turns}): {check_session_limits(agent.state)}")

# Test zone validation
print(f"\n🚦 Zone Transition Tests:")
print(f"   RED→GREEN: {agent._validate_zone_transition(Zone.GREEN, Zone.RED).value}")
print(f"   GREEN→RED: {agent._validate_zone_transition(Zone.RED, Zone.GREEN).value}")
print(f"   GREEN→YELLOW: {agent._validate_zone_transition(Zone.YELLOW, Zone.GREEN).value}")

# Test session export
export = agent.export_session()
print(f"\n💾 Session Export: {len(export['conversation_history'])} messages, "
      f"{len(export['zone_history'])} zone entries, "
      f"{len(export['tool_actions'])} tool actions")


## 8. Gradio Interactive Demo UI

A child-friendly chat interface with a live dashboard showing the student's state, tool invocations, zone stability, and session export.

In [ ]:
import gradio as gr

# Global agent instance
demo_agent = EduTutorAgent()


def chat_fn(message: str, history: list, profile: str, subject: str, topic: str):
    """Gradio chat handler (synchronous, local inference)."""
    demo_agent.state.profile = profile
    demo_agent.state.current_subject = subject
    demo_agent.state.current_topic = topic
    
    response = demo_agent.process_message(message)
    
    dashboard = demo_agent.get_dashboard_data()
    
    # Build rich dashboard display (Enhancement 3)
    tool_breakdown_str = ""
    if dashboard["tool_breakdown"]:
        tool_breakdown_str = " | ".join(
            f"`{k}`: {v}" for k, v in dashboard["tool_breakdown"].items()
        )
    else:
        tool_breakdown_str = "_none yet_"
    
    tools_this_turn = ", ".join(dashboard["tools_this_turn"]) if dashboard["tools_this_turn"] else "none"
    
    dashboard_text = (
        f"### 🎯 Zone\n"
        f"**Current:** {dashboard['zone']}\n\n"
        f"**Trail:** {dashboard['zone_trail']}\n\n"
        f"**Stability:** {dashboard['zone_stability']:.0%} "
        f"{'🟩' if dashboard['zone_stability'] >= 0.7 else '🟨' if dashboard['zone_stability'] >= 0.4 else '🟥'}\n\n"
        f"---\n"
        f"### 📈 Progress\n"
        f"**Mastery:** {dashboard['mastery']}\n\n"
        f"**Difficulty:** {dashboard['difficulty']}\n\n"
        f"**Success Rate:** {dashboard['success_rate']}\n\n"
        f"**Concepts:** {dashboard['concepts']}\n\n"
        f"---\n"
        f"### 🔧 Tools\n"
        f"**This turn:** {tools_this_turn}\n\n"
        f"**Total invocations:** {dashboard['tools_used']}\n\n"
        f"**Breakdown:** {tool_breakdown_str}\n\n"
        f"---\n"
        f"### ℹ️ Session\n"
        f"**Profile:** {dashboard['profile']}\n\n"
        f"**Turn:** {dashboard['turns']}\n\n"
        f"**Frustration streak:** {dashboard['consecutive_frustrations']}"
    )
    
    return response, dashboard_text


def reset_fn():
    """Reset the session."""
    global demo_agent
    demo_agent = EduTutorAgent()
    return [], "Session reset! 🔄"


def export_fn():
    """Export the current session as JSON."""
    try:
        export_data = demo_agent.export_session()
        filename = f"edututor_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        filepath = os.path.join(".", filename)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False)
        
        summary = (
            f"✅ Session exported to `{filename}`\n\n"
            f"- **Turns:** {export_data['total_turns']}\n"
            f"- **Messages:** {len(export_data['conversation_history'])}\n"
            f"- **Zone transitions:** {len(export_data['zone_history'])}\n"
            f"- **Tool actions:** {len(export_data['tool_actions'])}\n"
            f"- **Profile:** {export_data['profile']}\n"
            f"- **Subject:** {export_data['subject']} / {export_data['topic']}"
        )
        return summary
    except Exception as e:
        return f"❌ Export failed: {e}"


# Build the UI
with gr.Blocks(
    title="EduTutor — Neurodiversity-Affirming AI Tutor",
    theme=gr.themes.Soft(primary_hue="indigo"),
    css="""
        .dashboard { padding: 16px; border-radius: 12px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }
        .main-title { text-align: center; font-size: 2em; margin-bottom: 0.5em; }
    """
) as demo:
    gr.Markdown(
        """# 🧠 EduTutor
        ### A Neurodiversity-Affirming AI Tutor powered by Gemma 4
        *Running locally on GPU — no API key needed. Adapts to ADHD, Autism, Dyslexia, and Dyscalculia.*
        *Features: Zone validation, tool-informed responses, safety guardrails, session export.*"""
    )
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Chat with EduTutor",
                height=450,
                type="messages",
            )
            msg = gr.Textbox(
                label="Type your message",
                placeholder="Ask EduTutor for help with your schoolwork...",
                lines=2,
            )
            with gr.Row():
                send_btn = gr.Button("Send 📤", variant="primary")
                reset_btn = gr.Button("New Session 🔄")
                export_btn = gr.Button("Export Session 💾")
        
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Student Dashboard")
            dashboard_display = gr.Markdown(
                value="*Start chatting to see your dashboard!*",
                elem_classes=["dashboard"],
            )
            
            export_status = gr.Markdown(value="", visible=True)
            
            profile_dropdown = gr.Dropdown(
                label="Learner Profile",
                choices=["ADHD", "Autism", "Dyslexia", "Dyscalculia", "General"],
                value="ADHD",
            )
            subject_dropdown = gr.Dropdown(
                label="Subject",
                choices=["Math", "Reading", "Writing", "Science"],
                value="Math",
            )
            topic_input = gr.Textbox(
                label="Topic",
                value="Fractions",
                placeholder="e.g., Fractions, Phonics, Water Cycle",
            )
    
    # Wire up events
    def respond(message, history, profile, subject, topic):
        response, dashboard_text = chat_fn(message, history, profile, subject, topic)
        history = history or []
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response})
        return history, "", dashboard_text
    
    send_btn.click(
        respond,
        inputs=[msg, chatbot, profile_dropdown, subject_dropdown, topic_input],
        outputs=[chatbot, msg, dashboard_display],
    )
    msg.submit(
        respond,
        inputs=[msg, chatbot, profile_dropdown, subject_dropdown, topic_input],
        outputs=[chatbot, msg, dashboard_display],
    )
    reset_btn.click(reset_fn, outputs=[chatbot, dashboard_display])
    export_btn.click(export_fn, outputs=[export_status])


# Launch
demo.launch(share=True, debug=True)
print("\n🚀 EduTutor demo is live! Running on local GPU.")


## Summary

This notebook provides the **live demo** for the competition submission:

| Component | What It Does |
|---|---|
| **Zone Classifier** | Detects student emotional state (Green/Yellow/Red/Blue) |
| **Zone Validation** | Prevents impossible zone transitions (e.g., RED→GREEN forced through BLUE) |
| **Student State Machine** | Tracks mastery, difficulty, frustrations, concepts learned |
| **Tool System** | Flashcards, scaffolding hints, brain breaks, difficulty adjustment — **auto-invoked** and **fed back into LLM prompt** |
| **Safety Guardrails** | PII redaction, crisis escalation detection, session time limits |
| **Session Export** | One-click JSON export of full conversation, zones, and metrics |
| **ReAct Agent Loop** | Orchestrates classification → validation → tools → safety → response → state update |
| **Gradio UI** | Child-friendly chat interface with rich live dashboard |

**All inference runs locally on GPU — no API key needed.**

### Competition Scoring Alignment

| Criterion | Points | How This Notebook Addresses It |
|---|---|---|
| **Impact & Vision** | 40 | Neurodiversity-affirming design, Zones of Regulation framework, safety guardrails for vulnerable children |
| **Video Pitch** | 30 | Interactive Gradio demo with live dashboard showing zone transitions, tool invocations, and safety features |
| **Technical Depth** | 30 | Zone validation state machine, tool→LLM feedback loop, PII filtering, session export, modular architecture |

### Tool Invocation Log

The dashboard now shows **which tools were triggered** during each turn, with a full breakdown of invocation counts, zone stability metrics, and zone transition trails — making the agentic behavior fully observable for the demo video.

### Submission Checklist

- [ ] Record video demo showing all 4 emotional zones
- [ ] Write 1,500-word project write-up using metrics from Notebook 3
- [ ] Upload fine-tuned model to Hugging Face Hub
- [ ] Publish all 4 notebooks as public Kaggle notebooks
- [ ] Create public GitHub repository with README